In [ ]:
from pathlib import Path
import pickle
with open(Path("Data/c_elegans_preprocessed.pkl"), "rb") as f:
    data = pickle.load(f)
folds = data["folds"]

In [ ]:
import torch
from torch import nn


class CrossAttentionModel(nn.Module):
    """Predict whether a receptor-peptide pair is an agonist.

    Inputs are cached ESM-2 per-residue embeddings with shape
    (batch, padded_sequence_length, embedding_dim), plus true lengths.
    The 650M embeddings use embedding_dim=1280.
    """

    def __init__(self, embedding_dim=1280, num_heads=8, hidden_dim=256, dropout=0.1):
        super().__init__()
        self.receptor_norm = nn.LayerNorm(embedding_dim)
        self.peptide_norm = nn.LayerNorm(embedding_dim)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.attended_norm = nn.LayerNorm(embedding_dim)
        self.attended_dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(3 * embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    @staticmethod
    def masked_mean(values, padding_mask):
        # padding_mask is True where a row is padding.
        valid = (~padding_mask).unsqueeze(-1)
        return (values * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1)

    def forward(self, receptor_residues, peptide_residues, receptor_lengths, peptide_lengths):
        # Cached embeddings are float16; match the model's parameter dtype.
        dtype = self.cross_attention.in_proj_weight.dtype
        receptor_residues = receptor_residues.to(dtype=dtype)
        peptide_residues = peptide_residues.to(dtype=dtype)
        receptor_lengths = torch.as_tensor(receptor_lengths, device=receptor_residues.device)
        peptide_lengths = torch.as_tensor(peptide_lengths, device=peptide_residues.device)

        receptor_padding = (
            torch.arange(receptor_residues.size(1), device=receptor_residues.device)[None, :]
            >= receptor_lengths[:, None]
        )
        peptide_padding = (
            torch.arange(peptide_residues.size(1), device=peptide_residues.device)[None, :]
            >= peptide_lengths[:, None]
        )

        receptor = self.receptor_norm(receptor_residues)
        peptide = self.peptide_norm(peptide_residues)

        # Each peptide residue queries the receptor residues. Padded receptor
        # positions cannot receive attention. Padded peptide positions are
        # excluded when we summarize the attended sequence below.
        attended, _ = self.cross_attention(
            query=peptide,
            key=receptor,
            value=receptor,
            key_padding_mask=receptor_padding,
            need_weights=False,
        )
        attended = self.attended_norm(peptide + self.attended_dropout(attended))

        pair_features = torch.cat(
            [
                self.masked_mean(peptide, peptide_padding),
                self.masked_mean(receptor, receptor_padding),
                self.masked_mean(attended, peptide_padding),
            ],
            dim=-1,
        )
        # One raw logit per pair; the training code will choose the loss.
        return self.classifier(pair_features).squeeze(-1)

In [ ]:
for fold in folds:
     test_families = fold["test_families"]
     test = fold["test"]
     hyperparameter_tuning = fold["hyperparameter_tuning"]
     train = fold["train"]
     train_hpt = fold["train_hpt"]
    #!!! train cross attention model on esm2 here

    # prof said to use focal loss with ablation? "We will use focal loss, tuning the focus parameter, and then compare it against the weighted BCE so we can ensure that focal actually adds something other than simply reweighting."

    # first test on validation/training, tune model
    #then use test data
    #there will also be a test set for human and other organism's gpcr; haven't got around to importing it